In [ ]:
!pip install transformers accelerate torch torchvision gtts moviepy opencv-python bitsandbytes

from transformers import LlavaForConditionalGeneration, LlavaProcessor, pipeline
from moviepy.editor import VideoFileClip, AudioFileClip
from moviepy.editor import CompositeAudioClip
from gtts import gTTS
import torch
import cv2
import os
from PIL import Image
import shutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 9.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12


  if event.key is 'enter':



In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq
import numpy as np

model_id = "may-ur08/llava-commentary-gen"

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_8bit=True
)

The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



processor_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/470 [00:00<?, ?B/s]

video_preprocessor_config.json: 0.00B [00:00, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

In [ ]:
video_path = "t1.mp4"
video = VideoFileClip(video_path)

# Save video without audio
video_no_audio = video.without_audio()
video_no_audio.write_videofile("video_no_audio.mp4", codec="libx264")

# Extract 1 frame per second
fps = 4
frames = []
frame_dir = "frames"
os.makedirs(frame_dir, exist_ok=True)

cap = cv2.VideoCapture("video_no_audio.mp4")
count, saved = 0, 0
success = True

while success:
    success, image = cap.read()
    if success and int(cap.get(cv2.CAP_PROP_POS_FRAMES)) % int(cap.get(cv2.CAP_PROP_FPS)) == 0:
        frame_path = f"{frame_dir}/frame_{saved}.jpg"
        cv2.imwrite(frame_path, image)
        frames.append(frame_path)
        saved += 1
cap.release()

Moviepy - Building video video_no_audio.mp4.
Moviepy - Writing video video_no_audio.mp4



Moviepy - Done !
Moviepy - video ready video_no_audio.mp4


In [ ]:
import numpy as np
from collections import defaultdict

# --- rolling-window parameters ---
window = 4          # frames
stride = 1
fps_video = cap.get(cv2.CAP_PROP_FPS)   # we already released `cap`, so re-open briefly
cap = cv2.VideoCapture("video_no_audio.mp4")
fps_video = cap.get(cv2.CAP_PROP_FPS)
cap.release()

# --- helpers ---
def get_timestamp(frame_idx):
    return frame_idx / fps_video

event_log = []           # (timestamp, simple_event)
vote_buffer = []         # stores (frame_idx, raw_caption)

# --- main rolling loop ---
total_frames = len(frames)
for idx in range(0, total_frames - window + 1, stride):
    slice_paths = frames[idx: idx + window]

    # majority vote on the *central* frame of the window
    central_idx = idx + window // 2

    # run VLM on central frame only (you can average, but this is simplest)
    image = Image.open(slice_paths[window//2]).convert("RGB")
    prompt = "<image>\nUSER: What cricket shot is being played in this image?\nASSISTANT:"
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=50)
    caption = processor.decode(output[0], skip_special_tokens=True).strip()
    if "ASSISTANT:" in caption:
        caption = caption.split("ASSISTANT:")[-1].strip()

    # ignore noisy
    if len(caption) < 10 or "please provide" in caption.lower():
        continue

    vote_buffer.append((central_idx, caption))

# --- 75 % rule on contiguous same-label windows ---
# group contiguous same labels
grouped = defaultdict(list)
prev = None
for idx, cap in vote_buffer:
    label = cap.lower()
    if label == prev:
        grouped[label][-1].append(idx)
    else:
        grouped[label].append([idx])
    prev = label

# build event log
for label, runs in grouped.items():
    for run in runs:
        total_run_frames = len(run)
        mark_idx = run[int(0.75 * (total_run_frames - 1))]
        mark_time = get_timestamp(mark_idx)
        simple_event = label
        event_log.append((mark_time, simple_event))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

llm_id = "may-ur08/lora_model_cricket_commentary"
tokenizer = AutoTokenizer.from_pretrained(llm_id)
llm = AutoModelForCausalLM.from_pretrained(
    llm_id,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True
)

llm_pipe = pipeline(
    "text-generation",
    model=llm,
    tokenizer=tokenizer,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.7
)

wps = 3.3   # words per second for Kyutai TTS

enhanced_lines = []
for ts, simple in event_log:
    prompt = f"Rewrite for commentary at {ts:.2f}s: {simple}"
    out = llm_pipe(prompt)[0]["generated_text"].strip()
    # remove prompt echo
    if "Rewrite for commentary" in out:
        out = out.split(":", 1)[-1].strip()
    enhanced_lines.append((ts, out))

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

adapter_config.json:   0%|          | 0.00/926 [00:00<?, ?B/s]

...

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
  warnings.warn(warning_msg)



model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
# Fast install (as given by you)
!pip install 'sphn<0.2'
!pip install --no-deps "moshi==0.2.7"

import numpy as np
import torch
from moshi.models.loaders import CheckpointInfo
from moshi.models.tts import DEFAULT_DSM_TTS_REPO, DEFAULT_DSM_TTS_VOICE_REPO, TTSModel
from IPython.display import display, Audio
from scipy.io.wavfile import write
import subprocess

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.1/107.1 kB 5.6 MB/s eta 0:00:00


In [ ]:
import torch, gc, logging, warnings
from moshi.models.loaders import CheckpointInfo
from moshi.models.tts import TTSModel

# --- 1. pick device & dtype dynamically ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype   = torch.float16 if device.type == "cuda" else torch.float32
print("TTS trying device:", device, dtype)

# --- 2. try GPU first, fallback to CPU on OOM ---
try:
    checkpoint_info = CheckpointInfo.from_hf_repo(DEFAULT_DSM_TTS_REPO)
    tts_model = TTSModel.from_checkpoint_info(
        checkpoint_info, n_q=32, temp=0.6, device=device, dtype=dtype
    )
except RuntimeError as e:          # catches CUDA OOM
    if "CUDA out of memory" in str(e):
        warnings.warn("GPU OOM – falling back to CPU for TTS")
        torch.cuda.empty_cache()
        gc.collect()
        device = torch.device("cpu")
        dtype  = torch.float32
        tts_model = TTSModel.from_checkpoint_info(
            checkpoint_info, n_q=32, temp=0.6, device=device, dtype=dtype
        )
    else:
        raise e

voice = "expresso/ex03-ex02_narration_001_channel1_674s.wav"
voice_path = tts_model.get_voice_path(voice)

# --- 3. build audio segments, respecting relative timing ---
all_texts = [txt for _, txt in enhanced_lines]
entries = tts_model.prepare_script(all_texts, padding_between=1)
condition_attrs = tts_model.make_condition_attributes(
    [voice_path] * len(all_texts), cfg_coef=2.0
)

pcms = []
def _on_frame(frame):
    if (frame != -1).all():
        pcm = tts_model.mimi.decode(frame[:, 1:, :]).cpu().numpy()
        pcms.append(np.clip(pcm[0, 0], -1, 1))

with tts_model.mimi.streaming(len(all_texts)):
    tts_model.generate([entries], [condition_attrs], on_frame=_on_frame)

audio = np.concatenate(pcms, axis=-1)

# --- 4. save ---
write('tts_output.wav', tts_model.mimi.sample_rate, audio)
subprocess.run([
    'ffmpeg', '-y',
    '-i', 'tts_output.wav',
    '-codec:a', 'libmp3lame',
    '-qscale:a', '2',
    'tts_output.mp3'
])
print("Saved --> tts_output.mp3")

In [ ]:
final_video_path = "final_video_with_commentary.mp4"

video = VideoFileClip(video_path).without_audio()
audio = AudioFileClip("tts_output.mp3")

# trim if TTS longer than video
if audio.duration > video.duration:
    audio = audio.subclip(0, video.duration)

final_video = video.set_audio(audio)
final_video.write_videofile(final_video_path, codec="libx264", audio_codec="aac")